# Real-World Incident Case Studies

**Module 02 · Failures and Robustness** · Notebook 1 of 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/02_Failures_and_Robustness/01_incident_case_studies.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Beginner | ~75 min | [Module 01 · Perception](../01_Perception/README.md) (SAE levels, sensors) | SAE J3016 (2021), ISO 21448:2022 (context) |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

By the end of this notebook, you will be able to:

1. Read an official investigation report (NTSB, NHTSA) and separate *probable cause*, *contributing factors* and press claims
2. Reconstruct the Uber ATG Tempe crash (2018) from the NTSB's recorded perception timeline
3. Explain the documented findings of the Tesla Williston (2016), Mountain View (2018) and Spring (2021) investigations, including a widely repeated myth
4. Describe the Cruise San Francisco incident (2023) and its regulatory consequences
5. Classify perception-related failures with a taxonomy that keeps ISO 26262, ISO 21448, ISO/PAS 8800 and ISO/SAE 21434 apart
6. Apply root-cause analysis (5 Whys, fishbone) and derive design lessons

## Setup and imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

# Okabe-Ito colour-blind-safe palette
OKABE = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', '#000000']
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
rng = np.random.default_rng(42)  # not used for any data here; kept for consistency

print(f"NumPy {np.__version__} | pandas {pd.__version__}")

---

## Why study automated-driving failures?

> "Those who cannot remember the past are condemned to repeat it." — George Santayana

Studying real incidents matters because it lets us:

1. **Learn from mistakes** and prevent recurrence
2. **Identify systematic issues** — patterns that repeat across companies and systems
3. **Improve designs** — architectures, monitoring, fallback behaviour
4. **Inform standards and regulation** — reporting duties, recalls, type approval
5. **Calibrate expectations** — what today's technology can and cannot do

### Lessons from aviation

Commercial aviation became one of the safest modes of transport largely through **independent accident investigation, mandatory reporting and industry-wide sharing of lessons**. Road-vehicle automation is younger and more fragmented, but the same instruments are emerging: independent investigations by the US National Transportation Safety Board (NTSB), mandatory crash reporting to NHTSA (since June 2021) and software recalls.

### How to read the sources in this notebook

| Source type | Example | What it can tell you |
|---|---|---|
| Independent investigation | NTSB highway accident reports (HAR) and briefs (HIR) | Factual record, **probable cause**, contributing factors, recommendations |
| Regulator records | NHTSA recall reports (Part 573), investigation resumes (PE/EA/RQ), Standing General Order data | What the manufacturer/regulator formally stated; scope (number of vehicles) |
| Company statements, press | Blog posts, news articles | Early, often incomplete; sometimes contradicted by the investigation |

A recurring lesson of this notebook: **early media narratives are frequently wrong** (see the Williston and Spring cases). Always prefer the primary source.

> **Terminology.** Most crashes below involve **SAE Level 2 driver assistance** (the human driver must supervise at all times), not "autonomous vehicles". Uber ATG and Cruise operated **automated driving systems (ADS)** — Uber's as a developmental system with a safety operator, Cruise's driverless.

In [ ]:
# Notable crashes, with the documented facts and the primary source for each.
av_incidents = pd.DataFrame([
    dict(Date='2016-05-07', System='Tesla Model S (Autopilot)', Location='Williston, FL',
         Operation='SAE Level 2 driver assistance', Fatal=True,
         Description='Tractor-trailer turned left across the car\'s path; car (TACC at 74 mph in a 65 mph zone) '
                     'struck and passed under the trailer; no braking or steering before impact',
         Official_finding='Truck driver\'s failure to yield + car driver\'s inattention due to overreliance on automation; '
                          'system not designed to identify crossing traffic; used on roads it was not designed for',
         Source='NTSB HAR-17/02', tags=['driver overreliance', 'ODD misuse', 'crossing traffic not in design']),
    dict(Date='2018-03-18', System='Uber ATG test vehicle (Volvo XC90)', Location='Tempe, AZ',
         Operation='Developmental ADS with safety operator', Fatal=True,
         Description='Pedestrian pushing a bicycle across the road outside a crosswalk was detected 5.6 s before impact '
                     'but repeatedly re-classified; no emergency braking by design',
         Official_finding='Operator distracted by personal cell phone (probable cause); contributing: ATG safety culture, '
                          'ADOT oversight, pedestrian crossing outside crosswalk',
         Source='NTSB HAR-19/03', tags=['operator inattention', 'classification instability',
                                        'specification gap', 'safety culture']),
    dict(Date='2018-03-23', System='Tesla Model X (Autopilot)', Location='Mountain View, CA',
         Operation='SAE Level 2 driver assistance', Fatal=True,
         Description='Autosteer steered the vehicle into a highway gore area at about 71 mph; struck a damaged, '
                     'non-functional crash attenuator',
         Official_finding='Autopilot system limitations + driver distraction likely from a cell phone game application '
                          '+ overreliance on Autopilot',
         Source='NTSB HAR-20/01 (HWY18FH011)', tags=['driver overreliance', 'lane-keeping limitation',
                                                     'infrastructure condition']),
    dict(Date='2019-12-29', System='Tesla Model 3', Location='Cloverdale, IN (I-70)',
         Operation='Unconfirmed (Autopilot engagement not publicly confirmed)', Fatal=True,
         Description='Car struck a fire truck parked at an earlier crash scene; passenger killed',
         Official_finding='No NTSB report; NHTSA sent a Special Crash Investigation team',
         Source='WVPE / AP, 31 Dec 2019', tags=['stationary emergency vehicle']),
    dict(Date='2021-04-17', System='Tesla Model S', Location='Spring, TX',
         Operation='Manual driving (Autopilot could not be engaged on that road)', Fatal=True,
         Description='Run-off-road crash and fire; early media claimed "no one was in the driver\'s seat"',
         Official_finding='Driver\'s excessive speed (67 mph in a 30 mph zone) and impairment (alcohol, sedating '
                          'antihistamines); evidence indicates the driver was in the driver\'s seat',
         Source='NTSB HIR-23/02', tags=['not an automation failure']),
    dict(Date='2023-10-02', System='Cruise robotaxi (Chevrolet Bolt)', Location='San Francisco, CA',
         Operation='Driverless ADS', Fatal=False,
         Description='Pedestrian thrown into the AV\'s path by a hit-and-run driver; AV braked but struck her, then '
                     'executed a pull-over manoeuvre that dragged her about 20 ft',
         Official_finding='Post-collision logic mis-characterised the collision as lateral and pulled over instead of '
                          'remaining stationary (recall 23E-086); incomplete crash reporting (NHTSA consent order)',
         Source='NHTSA recall 23E-086; NHTSA consent order (Sep 2024)',
         tags=['post-collision behaviour', 'specification gap', 'reporting/transparency']),
])
av_incidents['Date'] = pd.to_datetime(av_incidents['Date'])

display(av_incidents[['Date', 'System', 'Location', 'Operation', 'Fatal', 'Source']])

n_fatal = int(av_incidents['Fatal'].sum())
n_l2 = int(av_incidents['Operation'].str.contains('Level 2').sum())
print("\n--- Incident summary (derived from the table) ---")
for _, r in av_incidents.iterrows():
    print(f"  {r['Date']:%Y-%m-%d} | {r['System']:34s} | fatal: {str(r['Fatal']):5s} | {r['Operation']}")
print(f"\n{len(av_incidents)} incidents, {n_fatal} fatal; {n_l2} involved SAE Level 2 driver assistance, "
      f"{int(av_incidents['Operation'].str.contains('ADS').sum())} an automated driving system.")
print("Pennsylvania, July 2016 (Model X rollover) is deliberately omitted: no investigation report exists and "
      "Autopilot use was disputed.")

In [ ]:
# Timeline: one row per type of operation, marker shape = fatal / non-fatal
op_order = ['Manual driving (Autopilot could not be engaged on that road)',
            'Unconfirmed (Autopilot engagement not publicly confirmed)',
            'SAE Level 2 driver assistance',
            'Developmental ADS with safety operator',
            'Driverless ADS']
op_short = ['Manual driving', 'Engagement unconfirmed', 'Level 2 assistance',
            'Developmental ADS\n(safety operator)', 'Driverless ADS']
ypos = {op: i for i, op in enumerate(op_order)}

fig, ax = plt.subplots(figsize=(12, 5))
for _, r in av_incidents.iterrows():
    y = ypos[r['Operation']]
    ax.scatter(r['Date'], y, s=220 if r['Fatal'] else 140,
               marker='X' if r['Fatal'] else 'o',
               color=OKABE[5] if r['Fatal'] else OKABE[4], edgecolors='black', zorder=3)
    ax.annotate(f"{r['System'].split(' (')[0]}\n{r['Location']}", (r['Date'], y),
                xytext=(0, 12), textcoords='offset points', ha='center', fontsize=8)
ax.set_yticks(range(len(op_order)))
ax.set_yticklabels(op_short)
ax.set_ylim(-0.6, len(op_order) - 0.2)
ax.set_xlim(pd.Timestamp('2015-09-01'), pd.Timestamp('2024-06-30'))
ax.set_xlabel('Date of crash')
ax.set_ylabel('Type of operation')
ax.set_title('Notable automated-driving and driver-assistance crashes (2016–2023)')
ax.legend(handles=[Line2D([0], [0], marker='X', color='w', markerfacecolor=OKABE[5], markeredgecolor='k',
                          markersize=12, label='Fatal'),
                   Line2D([0], [0], marker='o', color='w', markerfacecolor=OKABE[4], markeredgecolor='k',
                          markersize=10, label='Non-fatal')], loc='upper left')
plt.tight_layout()
plt.show()

print("--- Timeline (text summary) ---")
for op, short in zip(op_order, op_short):
    sub = av_incidents[av_incidents['Operation'] == op]
    if len(sub):
        years = ', '.join(f"{d:%Y}" for d in sub['Date'])
        print(f"  {short.replace(chr(10), ' '):34s}: {len(sub)} crash(es) [{years}]")

---

## Case study 1: Uber ATG, Tempe, Arizona (18 March 2018)

### Incident overview (NTSB HAR-19/03)

**Date and time:** 18 March 2018, 9:58 p.m.
**Location:** N. Mill Avenue, Tempe, Arizona (posted limit **45 mph**; road lit by street lighting)
**Vehicle:** modified 2017 Volvo XC90 with Uber ATG's **developmental automated driving system**, one safety operator on board
**Outcome:** a 49-year-old pedestrian, pushing a bicycle across the road outside a crosswalk, was fatally injured. It is widely reported as the first pedestrian fatality involving a vehicle operated by an automated driving system.

### What happened?

1. **Pedestrian:** crossed mid-block at night, outside a crosswalk, pushing a bicycle
2. **Vehicle:** in ADS mode, approaching at about 44 mph and **39 mph at impact**
3. **Detection:** the ADS first detected the pedestrian **5.6 s before impact** (radar), then lidar
4. **Classification:** the object was classified as *vehicle*, *other* (unknown) and *bicycle* — never as a pedestrian. At each re-classification the tracking history was discarded, so the predicted path was reset to "static"
5. **Hazard recognition:** only **1.2 s before impact** did the ADS predict the object to be on its path
6. **No emergency braking by design:**
   (a) Volvo's factory collision-avoidance functions (City Safety FCW/AEB) were **automatically deactivated whenever the Uber ADS was engaged**;
   (b) the ADS itself was **not designed to brake hard for crash mitigation**. On recognising an emergency it entered a 1 s **"action suppression"** period, then issued an auditory alert and began only a gradual slowdown
7. **Safety operator:** visually distracted by a personal phone — phone records show a TV show was **streaming** for the whole trip; the operator took the steering wheel 0.02 s before impact and braked after it

In [ ]:
# Recorded ADS perception/decision timeline — transcribed from NTSB HAR-19/03, Table, pp. 15-16
# https://www.ntsb.gov/investigations/accidentreports/reports/har1903.pdf
uber_timeline = pd.DataFrame({
    't_to_impact_s': [-5.6, -5.2, -4.2, -3.9, -3.8, -2.6, -2.5, -1.5, -1.2, -0.2, -0.02],
    'speed_mph':     [44.3, 44.6, 44.8, 44.8, 44.7, 44.6, 44.6, 43.8, 43.2, 40.5, 39.0],
    'classification': ['Vehicle', 'Other', 'Vehicle', 'Vehicle', 'Vehicle/Other alternating',
                       'Bicycle', 'Bicycle', 'Other', 'Bicycle', 'Bicycle', '-'],
    'sensor':        ['radar', 'lidar', 'lidar', 'lidar', 'lidar', 'lidar', 'lidar', 'lidar', 'lidar', 'lidar', '-'],
    'predicted_path': ['none (not on path)', 'static (not on path)', 'static (not on path)',
                       'left through lane (not on path)', 'static / left lane (not on path)',
                       'static (not on path)', 'left through lane (not on path)',
                       'static, partly on path', 'on SUV path', 'on SUV path', '-'],
    'ads_action':    ['-', '-', '-', '-', '-', '-', '-', 'plans to steer around object',
                      'emergency recognised; 1 s action suppression starts',
                      'auditory alert + gradual slowdown', 'operator takes steering; ADS disengages'],
})
display(uber_timeline)

# Derived facts
cls = uber_timeline['classification'][:-1]
changes = int((cls != cls.shift()).sum() - 1)
t_detect = uber_timeline['t_to_impact_s'].iloc[0]
t_emerg = uber_timeline.loc[uber_timeline['ads_action'].str.contains('emergency'), 't_to_impact_s'].iloc[0]
print("--- Derived from the NTSB timeline ---")
print(f"  First detection          : {abs(t_detect):.1f} s before impact")
print(f"  Hazard recognised        : {abs(t_emerg):.1f} s before impact "
      f"({abs(t_detect - t_emerg):.1f} s after first detection)")
print(f"  Recorded class changes   : at least {changes} (the -3.8 to -2.7 s window alternated several times)")
print(f"  'Pedestrian' ever output : {'yes' if (cls == 'Pedestrian').any() else 'no'}")
print("  Each class change discarded the tracking history, so the path was predicted as 'static'.")

In [ ]:
# Visualise the timeline: classification (y) vs time to impact (x); marker = on-path prediction
cls_levels = ['Vehicle', 'Other', 'Bicycle']
plot_df = uber_timeline[uber_timeline['classification'].isin(cls_levels)]
on_path = plot_df['predicted_path'].str.contains('on SUV path|partly on path')

fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(plot_df['t_to_impact_s'], plot_df['classification'].map(cls_levels.index), color='grey', lw=1, zorder=1)
ax.scatter(plot_df.loc[~on_path, 't_to_impact_s'], plot_df.loc[~on_path, 'classification'].map(cls_levels.index),
           s=110, color=OKABE[4], edgecolors='k', zorder=3, label='predicted NOT on vehicle path')
ax.scatter(plot_df.loc[on_path, 't_to_impact_s'], plot_df.loc[on_path, 'classification'].map(cls_levels.index),
           s=140, marker='D', color=OKABE[5], edgecolors='k', zorder=3, label='predicted (partly) on vehicle path')
ax.axvspan(-3.8, -2.7, color=OKABE[1], alpha=0.25, label='vehicle/other alternating (-3.8 to -2.7 s)')
ax.axvspan(-1.2, -0.2, color=OKABE[0], alpha=0.30, label='action suppression (1 s, no braking)')
ax.axvline(-0.02, color='k', ls='--', lw=1)
ax.text(-0.07, 1.5, 'operator\nsteers\n(-0.02 s)', ha='right', fontsize=8)
ax.set_yticks(range(len(cls_levels)))
ax.set_yticklabels(cls_levels)
ax.set_xlim(-6, 0.3)
ax.set_xlabel('Time to impact (s)')
ax.set_ylabel('ADS object classification')
ax.set_title('Uber ATG Tempe 2018: recorded classification and path prediction (NTSB HAR-19/03)')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

print("--- Text summary ---")
for _, r in uber_timeline.iterrows():
    print(f"  t={r['t_to_impact_s']:6.2f} s | {r['speed_mph']:4.1f} mph | {r['classification']:24s} | "
          f"{r['predicted_path']:32s} | {r['ads_action']}")

### Findings (NTSB HAR-19/03)

**Probable cause:** the operator's failure to monitor the driving environment and the ADS because she was visually distracted throughout the trip by her personal cell phone.

**Contributing factors:** Uber ATG's (1) inadequate safety risk assessment procedures, (2) ineffective oversight of vehicle operators and (3) lack of adequate mechanisms for addressing operators' automation complacency — "all a consequence of its inadequate safety culture"; the impaired pedestrian's crossing outside a crosswalk; and the Arizona Department of Transportation's insufficient oversight of automated-vehicle testing.

**Technical observations from the report:**
- **Detection was adequate** (5.6 s, on a street-lit road); the failures lay in classification, prediction and response design
- The **system design did not include consideration for jaywalking pedestrians** — a pedestrian outside a crosswalk was not an expected object
- **Tracking history was discarded** at every re-classification, so path prediction restarted from "static"
- **Action suppression** (1 s, no alert) was introduced because of concern about false alarms — a classic false-positive versus false-negative trade-off resolved in favour of comfort
- Volvo's factory AEB was deactivated in ADS mode; the ADS did not brake for crash mitigation

**Regulatory context:** in 2018 there was no federal crash-reporting duty for automated vehicles. Since **29 June 2021**, NHTSA's **Standing General Order 2021-01** requires manufacturers and operators to report certain crashes involving ADS and SAE Level 2 systems (amended since).

### Lessons learned

**Design principles**
1. **Do not remove independent safety layers** (here, the factory AEB) without an equivalent replacement
2. **Specify for the real world:** pedestrians do cross mid-block — the specification gap is an ISO 21448 *functional insufficiency* (a specification insufficiency), not only a testing gap
3. **Keep object history across re-classification**; uncertainty about *what* an object is should not erase *where it is going*
4. **Fail-safe under uncertainty:** an unknown object on a converging path warrants slowing down

**Organisational**
1. **Safety culture and safety management** (risk assessment, operator oversight)
2. **Monitor safety operators** (driver-monitoring systems); plan for automation complacency
3. **Transparent reporting** and industry-wide learning

**Technical**
1. Multi-frame, history-preserving tracking and classification
2. Conservative behaviour when classification is unstable
3. Diverse sensing (the Uber vehicle already had lidar, radar and cameras — sensor diversity alone did not prevent the crash)

---

## Case study 2: Tesla Autopilot (SAE Level 2) investigations

Tesla's 2015–2018 vehicles with Autopilot used a **forward camera plus a forward radar** (and ultrasonic sensors) — not lidar, and not camera-only. Camera-only "Tesla Vision" was introduced only in 2021.

### Williston, Florida (7 May 2016) — NTSB HAR-17/02

**Scenario:** 2015 Model S with Traffic-Aware Cruise Control (TACC) and Autosteer engaged at **74 mph in a 65 mph zone** on a divided highway with at-grade intersections.
**Event:** a tractor-semitrailer turned left across the car's path; the car struck and passed under the trailer. The driver was killed.

**Documented findings:**
- The automated vehicle control system **"was not designed to, and did not, identify the truck crossing the car's path"**; the AEB required **both camera and radar** to detect an object before activating
- Autopilot was used on a road type for which it was **not designed**
- Of 37 minutes in Autopilot, the system detected the driver's hands on the wheel for only **25 seconds**; there was no braking or steering before impact
- **No evidence** was found that any of the driver's devices was in use at the time of the crash (the popular "watching a movie" story is **not** an NTSB finding)
- "White truck against a bright sky" was Tesla's public explanation in 2016; it is **not** the NTSB's finding

**Probable cause:** the truck driver's failure to yield, combined with the car driver's inattention due to overreliance on vehicle automation; contributing was the operational design, which permitted prolonged disengagement and use inconsistent with the manufacturer's guidance.

### Mountain View, California (23 March 2018) — NTSB HAR-20/01

**Scenario:** 2017 Model X on US-101 with TACC and Autosteer engaged.
**Event:** Autosteer steered the vehicle into the **gore area** of a left-exit ramp; it struck a crash attenuator at about **71 mph**. The attenuator had been **damaged in an earlier crash and was non-functional**, increasing severity. The driver was killed.

**Probable cause:** the Autopilot system steering the vehicle into the gore area *due to system limitations*, and the driver's lack of response due to distraction *likely from a cell phone game application* and overreliance on Autopilot.

### Spring, Texas (17 April 2021) — a myth-busting case, NTSB HIR-23/02

Early media reports claimed "no one was in the driver's seat" and implied Autopilot. The NTSB found:
- **Autopilot could not have been engaged** on that residential road (no lane markings); evidence indicated TACC was not engaged either
- The car accelerated to **67 mph in a 30 mph zone**; the driver's blood-alcohol concentration was 0.151 g/dL
- Security video, EDR data, the seat-belt status and steering-wheel damage indicate the **driver was in the driver's seat** and moved to the rear after the crash
- **Probable cause:** excessive speed and failure to control the car due to impairment — **not** an automation failure

### Cloverdale, Indiana (29 December 2019)

A 2019 Model 3 struck a fire truck parked at an earlier crash scene on I-70; the **passenger was killed**. Whether Autopilot was engaged has not been publicly confirmed. Crashes into stationary first-responder vehicles became the focus of NHTSA investigations PE21-020 and EA22-002, which led to **recall 23V-838** (December 2023, 2,031,220 vehicles; see the timeline below).

### Common patterns across the Tesla investigations

1. **Operational design and misuse:** Level 2 used outside its intended road types; weak driver engagement monitoring
2. **Human factors:** overreliance, complacency, mode confusion (Level 2 is not Level 4)
3. **Design limitations:** crossing traffic and some stationary objects not within the system's design; lane-keeping can be misled by road geometry and markings
4. **Infrastructure:** damaged roadside hardware increased severity (Mountain View)

In [ ]:
# Qualitative comparison of sensor modalities in the situations above.
print("Illustrative, qualitative table (typical strengths/limitations; not measurements of any product).")
sensor_limitations = pd.DataFrame({
    'Situation': ['Crossing trailer, bright sky', 'Stationary object in lane', 'Night, dark clothing',
                  'Heavy rain / fog', 'Low sun / glare'],
    'Camera': ['contrast-dependent', 'detects; classification-dependent', 'degraded (needs lighting)',
               'degraded', 'can saturate'],
    'Radar': ['detects; stationary/overhead returns often filtered by design', 'detects; often filtered as clutter',
              'unaffected by darkness', 'robust', 'unaffected'],
    'LiDAR': ['geometry detected', 'geometry detected', 'unaffected by darkness (active sensor)',
              'degraded (scattering)', 'mostly unaffected (sunlight adds noise)'],
})
display(sensor_limitations)

print("--- Text summary ---")
for _, r in sensor_limitations.iterrows():
    print(f"  {r['Situation']:30s} | camera: {r['Camera']:34s} | radar: {r['Radar']}")
print("\nSensor diversity reduces common-cause perception failures, but it cannot compensate for a function that")
print("was never designed for the scenario (Williston) or for a missing specification (Tempe, which had lidar).")

---

## Case study 3: Cruise, San Francisco (2 October 2023)

### Incident overview

**Vehicle:** Cruise driverless robotaxi (ADS, no one on board responsible for driving)
**Sequence (NHTSA recall report 23E-086; NHTSA consent order):**
1. A **human-driven vehicle** in the adjacent lane struck a pedestrian and threw her into the Cruise AV's path (hit-and-run)
2. The Cruise AV **braked hard but struck the pedestrian**, coming to a stop with her underneath the vehicle
3. Its collision-detection subsystem **mis-characterised the event as a lateral (side) collision** and commanded a **pull-over manoeuvre** "out of traffic" instead of remaining stationary
4. The pull-over **dragged the pedestrian about 20 feet**

### Root causes (as documented)

**1. Behaviour specification gap (post-collision):**
- The post-collision response did not account for a person trapped under or immediately adjacent to the vehicle before moving
- "Pull over after a collision" is reasonable in general but unsafe in this situation — context-dependent behaviour was missing

**2. Reporting and transparency:**
- Cruise's initial crash report to NHTSA under the Standing General Order **omitted the dragging**; regulators learned of it from video
- The California DMV cited **misrepresentation** of safety information as one ground for suspension

**3. Safety process:**
- Multi-stage (secondary) collision scenarios were not adequately covered in the safety case

### Regulatory and corporate consequences (as of October 2026)

| Date | Event | Source |
|---|---|---|
| 24 Oct 2023 | California DMV **suspends** Cruise's driverless testing and deployment permits ("not safe for the public's operation"; misrepresentation) | [CA DMV statement](https://www.dmv.ca.gov/portal/news-and-media/dmv-statement-on-cruise-llc-suspension/) |
| 7 Nov 2023 | **Recall 23E-086**: 950 driverless vehicles; software update so the AV remains stationary in such cases | [NHTSA Part 573 report](https://static.nhtsa.gov/odi/rcl/2023/RCLRPT-23E086-7725.PDF) |
| 30 Sep 2024 | NHTSA **consent order**, **$1.5 M** penalty for incomplete crash reports | [NHTSA press release](https://www.nhtsa.gov/press-releases/consent-order-cruise-crash-reporting) |
| 14 Nov 2024 | Deferred prosecution agreement; Cruise admits a false report to NHTSA; **$500,000** fine | [TechCrunch](https://techcrunch.com/2024/11/15/gms-cruise-to-pay-500000-fine-to-doj-admits-submitting-false-report) |
| 10 Dec 2024 | **GM stops funding Cruise's robotaxi business** and refocuses on driver assistance | [Fortune](https://fortune.com/2024/12/10/general-motors-stop-funding-cruise-robotaxi-business) |

### Lessons learned

1. **Specify post-event behaviour:** after a collision, the safe action may be to *stay still*
2. **Perception coverage around the vehicle** — including close-range, low-lying and partly occluded road users (see also Zoox recall 25E-037 below)
3. **Context-aware fallback:** "pull over" is not always the minimal-risk manoeuvre
4. **Transparency is a safety mechanism:** incomplete reporting cost Cruise its permits and, ultimately, its business
5. **Test multi-stage incidents**, not only single events

---

## 2021–2026: recalls and investigations

Since 2021, the main instruments in the US have been **mandatory crash reporting** (Standing General Order), **defect investigations** (preliminary evaluation PE → engineering analysis EA; recall query RQ) and **software recalls**, mostly delivered over the air. The table lists verified events; statuses are **as of October 2026** and may have changed since.

In [ ]:
# Verified recalls and investigations (as of October 2026). Population = vehicles covered.
events = pd.DataFrame([
    ('2021-06-29', 'NHTSA', 'Standing General Order 2021-01: crash reporting for ADS and SAE Level 2 ADAS', None,
     'https://www.nhtsa.gov/sites/nhtsa.gov/files/2021-06/Standing_General_Order_2021_01-digital-06292021.pdf'),
    ('2023-11-07', 'Cruise', 'Recall 23E-086: post-collision pull-over logic (pedestrian dragged)', 950,
     'https://static.nhtsa.gov/odi/rcl/2023/RCLRPT-23E086-7725.PDF'),
    ('2023-12-12', 'Tesla', 'Recall 23V-838: Autosteer controls may be insufficient to prevent misuse (from PE21-020/EA22-002)', 2031220,
     'https://static.nhtsa.gov/odi/rcl/2023/RCLRPT-23V838-8276.PDF'),
    ('2024-04-25', 'Tesla', 'Recall query RQ24-009: adequacy of the 23V-838 remedy', 2031220,
     'https://static.nhtsa.gov/odi/inv/2024/INOA-RQ24009-12046.pdf'),
    ('2024-06-10', 'Waymo', 'Recall 24E-049: software/map update after low-speed collision with a pole (Phoenix, May 2024)', 672,
     'https://www.nbcnews.com/tech/tech-news/waymo-recalls-software-cars-robotaxi-crash-rcna157030'),
    ('2024-10-17', 'Tesla', 'PE24-031: FSD collisions in reduced roadway visibility (sun glare, fog, dust)', 2410002,
     'https://static.nhtsa.gov/odi/inv/2024/INOA-PE24031-23232.pdf'),
    ('2025-05-01', 'Zoox', 'Recall 25E-029: prediction of other vehicles at > 40 mph (after Las Vegas collision, 8 Apr 2025)', 270,
     'https://static.nhtsa.gov/odi/rcl/2025/RCLRPT-25E029-4731.PDF'),
    ('2025-05-22', 'Zoox', 'Recall 25E-037: may resume travel with a prone road user immediately adjacent', 270,
     'https://static.nhtsa.gov/odi/rcl/2025/RCLRPT-25E037-4912.pdf'),
    ('2025-10-07', 'Tesla', 'PE25-012: FSD traffic-law violations (red lights, wrong direction of travel)', 2882566,
     'https://static.nhtsa.gov/odi/inv/2025/INOA-PE25012-19171.pdf'),
    ('2025-12-10', 'Waymo', 'Recall 25E-084: vehicles may pass a stopped school bus', 3067,
     'https://static.nhtsa.gov/odi/rcl/2025/RCLRPT-25E084-7732.pdf'),
    ('2026-03-18', 'Tesla', 'EA26-002 (upgrade of PE24-031): FSD degradation detection in reduced visibility', 3203754,
     'https://static.nhtsa.gov/odi/inv/2026/INOA-EA26002-10023.pdf'),
], columns=['Date', 'Organisation', 'Event', 'Vehicles', 'Source'])
events['Date'] = pd.to_datetime(events['Date'])
display(events[['Date', 'Organisation', 'Event', 'Vehicles']])

orgs = ['NHTSA', 'Tesla', 'Cruise', 'Waymo', 'Zoox']
fig, ax = plt.subplots(figsize=(12, 4.5))
for i, org in enumerate(orgs):
    sub = events[events['Organisation'] == org]
    ax.scatter(sub['Date'], [i] * len(sub), s=120, color=OKABE[i % len(OKABE)], edgecolors='k', zorder=3)
    for k, (_, r) in enumerate(sub.iterrows()):
        label = r['Event'].split(':')[0].split(' (')[0].replace('Recall query ', '').replace('Recall ', '')
        ax.annotate(label, (r['Date'], i), xytext=(0, 10 if k % 2 == 0 else -16), textcoords='offset points',
                    ha='center', fontsize=8)
ax.set_yticks(range(len(orgs)))
ax.set_yticklabels(orgs)
ax.set_ylim(-0.6, len(orgs) - 0.3)
ax.set_xlabel('Date')
ax.set_ylabel('Organisation')
ax.set_title('Recalls and NHTSA investigations involving driving automation, 2021–2026 (as of Oct 2026)')
plt.tight_layout()
plt.show()

print("--- Text summary ---")
for _, r in events.iterrows():
    n = f"{int(r['Vehicles']):,} vehicles" if pd.notna(r['Vehicles']) else 'n/a'
    print(f"  {r['Date']:%Y-%m-%d} | {r['Organisation']:6s} | {n:>18s} | {r['Event']}")
ads = events[events['Organisation'].isin(['Cruise', 'Waymo', 'Zoox'])]
print(f"\n{len(ads)} of the {len(events)} events concern driverless ADS fleets; all of their remedies were "
      "software (and map) updates.")

---

## Failure taxonomy for automated-driving perception

A systematic classification of how perception can fail. The last column shows which standard primarily governs each failure type — keeping these boundaries sharp is essential later in the course:

- **ISO 26262** — malfunctioning behaviour caused by **E/E faults** (random hardware, systematic software/hardware faults)
- **ISO 21448 (SOTIF)** — hazards from **functional insufficiencies** (specification or performance insufficiencies) activated by **triggering conditions**, plus reasonably foreseeable misuse
- **ISO/PAS 8800** — AI/ML-specific insufficiencies (data, model, generalisation)
- **ISO/SAE 21434** — intentional attacks

In [ ]:
taxonomy = pd.DataFrame([
    ('Detection', 'False negative (miss)', 'Object present but not detected', 'ISO 21448 / ISO/PAS 8800'),
    ('Detection', 'False positive (phantom)', 'Non-existent object reported (e.g. phantom braking)', 'ISO 21448 / ISO/PAS 8800'),
    ('Detection', 'Late detection', 'Detected too late for a safe response', 'ISO 21448'),
    ('Classification', 'Misclassification', 'Wrong class (e.g. pedestrian as "other")', 'ISO 21448 / ISO/PAS 8800'),
    ('Classification', 'Classification instability', 'Label oscillates between frames (Tempe 2018)', 'ISO 21448 / ISO/PAS 8800'),
    ('Classification', 'Unknown class', 'Object type absent from training data', 'ISO/PAS 8800'),
    ('Tracking/prediction', 'Lost track / history reset', 'Track discarded, e.g. on re-classification', 'ISO 21448'),
    ('Tracking/prediction', 'ID switch', 'Identities swapped between objects', 'ISO 21448'),
    ('Tracking/prediction', 'Trajectory error', 'Wrong motion prediction', 'ISO 21448'),
    ('Localisation/map', 'Position error', 'Wrong ego or object position', 'ISO 21448'),
    ('Localisation/map', 'Map mismatch', 'Outdated or incorrect map (e.g. road edge not defined)', 'ISO 21448'),
    ('Localisation/map', 'GNSS spoofing', 'Deliberately falsified positioning signal', 'ISO/SAE 21434'),
    ('Sensor', 'Hardware fault', 'Camera, LiDAR or radar malfunction', 'ISO 26262'),
    ('Sensor', 'Calibration drift', 'Extrinsic calibration degrades over time', 'ISO 26262 / ISO 21448'),
    ('Sensor', 'Environmental degradation', 'Rain, fog, glare, darkness (triggering conditions)', 'ISO 21448'),
    ('Sensor', 'Contamination/occlusion', 'Dirt, water film, obstruction', 'ISO 21448'),
    ('Fusion', 'Conflicting sensors', 'Camera says "car", LiDAR says "empty"', 'ISO 21448'),
    ('Fusion', 'Timing fault', 'Late or desynchronised sensor data', 'ISO 26262'),
    ('Fusion', 'Fusion logic insufficiency', 'Wrong weighting or confirmation rule', 'ISO 21448'),
    ('System', 'Software bug', 'Code defect causes incorrect behaviour', 'ISO 26262'),
    ('System', 'Use outside ODD / misuse', 'Operated outside its design domain', 'ISO 21448 (foreseeable misuse)'),
    ('System', 'Adversarial input', 'Manipulated sign, sensor spoofing', 'ISO/SAE 21434'),
], columns=['Category', 'Failure_type', 'Description', 'Primary_standard'])

print("AV PERCEPTION FAILURE TAXONOMY\n" + "=" * 70)
for cat, grp in taxonomy.groupby('Category', sort=False):
    print(f"\n{cat.upper()}")
    for _, r in grp.iterrows():
        print(f"  - {r['Failure_type']}: {r['Description']}  [{r['Primary_standard']}]")

In [ ]:
# Count failure types per category, coloured by the (first-named) primary standard
taxonomy['Std_main'] = taxonomy['Primary_standard'].str.split(' /| \\(', regex=True).str[0].str.strip()
counts = taxonomy.groupby(['Category', 'Std_main'], sort=False).size().unstack(fill_value=0)
std_cols = {'ISO 26262': OKABE[4], 'ISO 21448': OKABE[0], 'ISO/PAS 8800': OKABE[2], 'ISO/SAE 21434': OKABE[5]}
counts = counts[[s for s in std_cols if s in counts.columns]]

ax = counts.plot(kind='barh', stacked=True, figsize=(10, 5),
                 color=[std_cols[c] for c in counts.columns], edgecolor='k')
ax.set_xlabel('Number of failure types in the taxonomy')
ax.set_ylabel('Failure category')
ax.set_title('Perception failure taxonomy by category and primary standard')
ax.invert_yaxis()
ax.legend(title='Primary standard', fontsize=9)
plt.tight_layout()
plt.show()

print("--- Text summary ---")
for std in counts.columns:
    print(f"  {std:14s}: {int(counts[std].sum())} failure types")
print(f"  Total: {len(taxonomy)} failure types in {taxonomy['Category'].nunique()} categories")

---

## Root-cause analysis

### The "5 Whys" — Uber ATG, Tempe

1. **Why did the collision occur?**
   The vehicle did not brake in time, and the operator did not intervene.
2. **Why did the ADS not brake?**
   It recognised a hazard only 1.2 s before impact, then (by design) suppressed action for 1 s and was not designed to brake hard for crash mitigation; the factory AEB was deactivated.
3. **Why so late?**
   The object's class kept changing, and each change discarded its tracking history, so its path was predicted as "static" until it was almost in the lane.
4. **Why was the design like this?**
   The specification did not include pedestrians crossing outside crosswalks, and action suppression was chosen to avoid false alarms.
5. **Why were these design decisions not caught?**
   ATG had no adequate safety risk assessment and relied on a safety operator without effective oversight or complacency countermeasures (NTSB: inadequate safety culture).

**Root cause:** a **specification insufficiency** (an ISO 21448 functional insufficiency) together with **organisational** failures in safety management — not merely "insufficient testing".

### Fishbone (Ishikawa) categories

- **People:** operator distraction (phone streaming), automation complacency
- **Process:** no adequate safety risk assessment, weak operator oversight
- **Technology:** classification instability, history reset, action suppression, AEB deactivated
- **Environment:** night (street-lit), mid-block crossing
- **Management/regulation:** safety culture; limited state oversight of testing

In [ ]:
root_cause_factors = pd.DataFrame({
    'Category': ['Technical', 'Technical', 'Technical', 'Technical', 'Specification',
                 'Human', 'Organisational', 'Organisational', 'Environmental', 'Regulatory'],
    'Contributing_factor': [
        'Classification instability (vehicle/other/bicycle)',
        'Tracking history discarded on re-classification',
        '1 s action suppression; no crash-mitigation braking',
        'Volvo factory AEB deactivated in ADS mode',
        'Design did not consider pedestrians outside crosswalks',
        'Operator visually distracted by phone',
        'Inadequate safety risk assessment',
        'Ineffective operator oversight / no complacency countermeasures',
        'Impaired pedestrian crossing outside crosswalk at night',
        'Insufficient state oversight of AV testing',
    ],
    'NTSB_role': ['Finding', 'Finding', 'Finding', 'Finding', 'Finding',
                  'Probable cause', 'Contributing', 'Contributing', 'Contributing', 'Contributing'],
    'Mitigation': [
        'Multi-frame classification with uncertainty',
        'Preserve object history across class changes',
        'Brake for mitigation when collision is unavoidable',
        'Keep independent safety layers active',
        'Specify foreseeable road-user behaviour (ISO 21448)',
        'Driver-monitoring system with alerts',
        'Safety management system with hazard analysis',
        'Operator monitoring, two-person crews, training',
        'Assume mid-block crossings in urban ODD',
        'Mandatory reporting and permit conditions',
    ],
})
display(root_cause_factors)

print("--- Root-cause analysis: Uber ATG Tempe (text summary) ---")
for _, r in root_cause_factors.iterrows():
    print(f"  [{r['NTSB_role']:14s}] {r['Category']:14s} | {r['Contributing_factor']}")
    print(f"  {'':17s}Mitigation: {r['Mitigation']}")
role_counts = root_cause_factors['NTSB_role'].value_counts()
print("\nFactors by NTSB role: " + ", ".join(f"{k}: {v}" for k, v in role_counts.items()))

---

## Common patterns across automated-driving failures

### Technical patterns

1. **Sensor limitations:**
   - Camera: lighting (glare, darkness), weather, contrast
   - LiDAR: rain, snow, fog (scattering)
   - Radar: low angular resolution; stationary returns often filtered
2. **Perception challenges:**
   - Stationary objects in the lane (trucks, barriers, emergency vehicles)
   - Unusual road users, poses or behaviours (mid-block crossing)
   - Occlusion and close-range blind spots
   - Out-of-distribution objects (Notebook 03)
3. **System design issues:**
   - Removed or missing independent safety layers
   - Functions used outside their intended operational design domain (ODD)
   - Specification gaps for foreseeable situations (post-collision behaviour, crossing traffic)

### Human factors

1. **Overreliance:** misunderstanding capabilities, complacency, attention decline
2. **Mode confusion:** not knowing which mode is active; Level 2 treated as Level 4
3. **Foreseeable misuse:** must be addressed by design (ISO 21448), not only by owner's manuals

### Organisational issues

1. **Development pressure:** speed over safety
2. **Safety culture:** oversight, transparency, learning from near-misses

In [ ]:
# Frequency of failure patterns, computed from the tags in the incident table above
# (excluding Spring, TX, which was not an automation failure)
tagged = av_incidents[~av_incidents['tags'].apply(lambda t: 'not an automation failure' in t)]
freq = tagged.explode('tags')['tags'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(freq.index, freq.values, color=OKABE[4], edgecolor='k')
ax.set_xlabel(f'Number of incidents with this pattern (n = {len(tagged)} incidents)')
ax.set_ylabel('Failure pattern (tag)')
ax.set_title('Failure patterns in the incident table (small sample, tags assigned by the author)')
ax.xaxis.get_major_locator().set_params(integer=True)
plt.tight_layout()
plt.show()

print("--- Failure patterns (text summary, derived from the tags) ---")
for tag, n in freq.sort_values(ascending=False).items():
    print(f"  {n} x {tag}")
top = freq[freq == freq.max()].index.tolist()
print(f"\nMost frequent pattern(s) in this sample: {', '.join(top)}.")
print(f"Caution: {len(tagged)} hand-picked incidents are not a statistical sample.")

---

## Safety perspective

The incidents in this notebook map onto the safety standards studied later in the course:

| Observation | Where it is treated |
|---|---|
| Pedestrians outside crosswalks not in the specification (Tempe); post-collision behaviour (Cruise) | **Specification insufficiencies** and **triggering conditions** in ISO 21448 — [04 · SOTIF fundamentals](../04_SOTIF_ISO21448/01_sotif_fundamentals.ipynb), [04 · Triggering conditions and scenarios](../04_SOTIF_ISO21448/02_triggering_conditions_and_scenarios.ipynb) |
| Autopilot used outside its intended road types; driver overreliance | **Reasonably foreseeable misuse** (ISO 21448) and ODD monitoring — [07 · ODD and runtime monitoring](../07_Runtime_Assurance_and_Operations/01_odd_and_runtime_monitoring.ipynb) |
| Classification instability, unknown object types | ML-specific insufficiencies (ISO/PAS 8800) — [06 · AI safety standards](../06_AI_Safety_ISO_PAS_8800/01_ai_safety_standards.ipynb), [06 · Uncertainty types](../06_AI_Safety_ISO_PAS_8800/02_uncertainty_types.ipynb) |
| Factory AEB deactivated; independent safety layers | Safety architecture, ISO 26262 safety mechanisms — [03 · ISO 26262 fundamentals](../03_Functional_Safety_ISO26262/01_iso26262_fundamentals.ipynb) |
| Crash reporting, recalls, permits | [08 · Regulation and type approval](../08_Standards_Regulation_and_Safety_Case/02_regulation_and_type_approval.ipynb) |

---

## Exercise: failure analysis

**Scenario:** an automated shuttle on a university campus (driverless ADS, 25 km/h maximum) has an incident:

- **Conditions:** early morning, light fog, wet road
- **Event:** a student on an electric scooter crosses in front of the shuttle at an intersection
- **Result:** the shuttle does not brake; the student swerves and avoids a collision
- **System logs:** object detected 2.5 s before the conflict point, classified as "bicycle" with 45% confidence

**Your task:**

1. **Classify the failure** using the taxonomy above (and name the primary standard)
2. **Apply the 5 Whys** to find a root cause
3. **Identify contributing factors** (technical, specification, process, environmental)
4. **Propose mitigations** for each factor
5. **State the lessons** for system design

In [ ]:
# TODO: complete your analysis
your_analysis = {
    'failure_classification': '',  # e.g. "Classification: misclassification (ISO 21448 / ISO/PAS 8800)"
    'five_whys': [
        # add your five why-questions and answers
    ],
    'contributing_factors': {
        'technical': [],
        'specification': [],
        'process': [],
        'environmental': [],
    },
    'mitigations': [],
    'lessons_learned': [],
}

print("Your failure analysis:")
print(f"\nClassification: {your_analysis['failure_classification'] or '(not yet filled in)'}")
print("\n5 Whys:")
for i, why in enumerate(your_analysis['five_whys'], 1):
    print(f"  {i}. {why}")
print("\nReflection:")
print("  - Could this incident have been prevented by design rather than by testing?")
print("  - Which triggering conditions (ISO 21448) are present?")
print("  - Would a 45% confidence detection justify braking? What does that imply for false alarms?")

---

## Key takeaways

### What the investigations actually found
- **Uber ATG (2018):** detection 5.6 s before impact; failures in classification/prediction design, deliberate removal of emergency braking, an unmonitored distracted operator and an inadequate safety culture
- **Tesla Autopilot (2016, 2018):** Level 2 systems used outside their design, driver overreliance, design limitations (crossing traffic, gore areas); the vehicles had camera **and** radar
- **Spring, TX (2021):** not an automation crash — a reminder to check primary sources
- **Cruise (2023):** a post-collision behaviour specification gap, followed by incomplete reporting with severe regulatory and business consequences
- **2023–2026:** software recalls and investigations (Tesla, Cruise, Waymo, Zoox) are now the main feedback loop

### Failure taxonomy
- Seven categories: detection, classification, tracking/prediction, localisation/map, sensor, fusion, system
- Each failure type has a primary standard: ISO 26262 (faults), ISO 21448 (insufficiencies, triggering conditions, misuse), ISO/PAS 8800 (ML), ISO/SAE 21434 (attacks)

### Root-cause analysis
- **5 Whys** and **fishbone** diagrams push past the first technical symptom to specification and organisational causes

### Design principles from failures
1. Do not remove independent safety layers
2. Specify foreseeable behaviour of other road users and post-event behaviour
3. Behave conservatively under perception uncertainty
4. Design against foreseeable misuse (driver monitoring, ODD enforcement)
5. Safety culture and transparent reporting

---

## Self-assessment

1. In the Uber ATG Tempe crash, why did the repeated changes of object classification delay hazard recognition?
2. The NTSB found the Williston Autopilot system was not designed for crossing traffic. Which ODD restriction or design change (sensor, function or driver monitoring) would have addressed this?
3. Using the failure taxonomy, classify the Cruise San Francisco incident. Which categories apply and why?
4. Apply the 5 Whys to the Mountain View crash. What root cause do you arrive at?
5. Why is the Spring, Texas crash a useful teaching case even though no automation was involved?

<details>
<summary><b>Show answers</b></summary>

1. Each re-classification was treated as a new object without tracking history, so its path was predicted as "static" and not on the vehicle's path. Only 1.2 s before impact was the object predicted to be on the path — too late, especially as action suppression then delayed any response by 1 s.
2. Restricting Autosteer/TACC to roads for which it was designed (limited-access highways without crossing traffic), i.e. enforcing the ODD; effective driver-engagement monitoring; and designing perception and AEB for crossing traffic. Adding a sensor alone would not have helped if the function never considered crossing vehicles.
3. Primarily a **system-level/behaviour specification** failure (post-collision response), with **detection/tracking** aspects (a person under or immediately adjacent to the vehicle). In ISO 21448 terms: a specification insufficiency activated by a rare triggering condition (a secondary collision caused by another vehicle).
4. Example: Why did the car hit the attenuator? It steered into the gore area. Why? Autosteer followed the lane geometry/markings into the gore (system limitation). Why was this not corrected? The driver was distracted (likely a phone game) and over-relied on Autopilot. Why possible? Driver engagement monitoring allowed it. Why was the outcome fatal? The attenuator was damaged and not repaired. Root cause: system limitation combined with insufficient protection against foreseeable misuse; contributing: infrastructure maintenance.
5. It shows how early narratives ("no one in the driver's seat", "Autopilot") can be wrong. Safety lessons must be drawn from evidence (EDR data, video, testing), otherwise effort goes into the wrong countermeasures.

</details>

---

## References

**Investigation reports and regulator records**
- NTSB (2017). *Collision Between a Car Operating With Automated Vehicle Control Systems and a Tractor-Semitrailer Truck Near Williston, Florida, May 7, 2016*. HAR-17/02. https://www.ntsb.gov/investigations/accidentreports/reports/har1702.pdf
- NTSB (2019). *Collision Between Vehicle Controlled by Developmental Automated Driving System and Pedestrian, Tempe, Arizona, March 18, 2018*. HAR-19/03 (case HWY18MH010). https://www.ntsb.gov/investigations/accidentreports/reports/har1903.pdf
- NTSB (2020). *Collision Between a Sport Utility Vehicle Operating With Partial Driving Automation and a Crash Attenuator, Mountain View, California, March 23, 2018*. HAR-20/01 (HWY18FH011). https://www.ntsb.gov/investigations/Pages/HWY18FH011.aspx
- NTSB (2023). *Electric Vehicle Run-Off-Road Crash and Postcrash Fire, Spring, Texas, April 17, 2021*. HIR-23/02. https://www.ntsb.gov/investigations/AccidentReports/Reports/HIR2302.pdf
- NHTSA (2021). *Standing General Order 2021-01*. https://www.nhtsa.gov/sites/nhtsa.gov/files/2021-06/Standing_General_Order_2021_01-digital-06292021.pdf
- NHTSA recall and investigation documents as linked in the 2021–2026 table (23E-086, 23V-838, RQ24-009, PE24-031, 25E-029, 25E-037, PE25-012, 25E-084, EA26-002).
- California DMV (24 Oct 2023). *DMV statement on Cruise LLC suspension*. https://www.dmv.ca.gov/portal/news-and-media/dmv-statement-on-cruise-llc-suspension/
- NHTSA (30 Sep 2024). *Consent order: Cruise crash reporting*. https://www.nhtsa.gov/press-releases/consent-order-cruise-crash-reporting

**Secondary sources (no investigation report available)**
- WVPE / AP (31 Dec 2019). *Tesla rear-ends fire truck in Indiana, killing Arizona woman*. https://www.wvpe.org/indiana-news/2019-12-31/tesla-rear-ends-fire-truck-in-indiana-killing-arizona-woman
- NBC News (June 2024). *Waymo recalls software in all its cars after its robotaxi crashes into a pole*. https://www.nbcnews.com/tech/tech-news/waymo-recalls-software-cars-robotaxi-crash-rcna157030
- TechCrunch (15 Nov 2024); Fortune (10 Dec 2024) — as linked in the Cruise table.

**Standards**
- SAE J3016 (2021). *Taxonomy and Definitions for Terms Related to Driving Automation Systems for On-Road Motor Vehicles*.
- ISO 21448:2022. *Road vehicles — Safety of the intended functionality*.

**Related work by the author**
- Patel, M., Jung, R. (2023). Simulation-Based Application of Safety of the Intended Functionality to Mitigate Foreseeable Misuse in Automated Driving Systems. SAE. https://doi.org/10.47953/sae-pp-00371
- Patel, M., Jung, R., Cakir, Y. (2023). Simulation-Based Testing of Foreseeable Misuse by the Driver Applicable for Highly Automated Driving. LNNS, Springer. https://doi.org/10.1007/978-981-99-3043-2_15
- Patel, M., Jung, R., Khatun, M. (2025). A Systematic Literature Review on Safety of the Intended Functionality for Automated Driving Systems. SAE Technical Paper 2025-01-5030. https://doi.org/10.4271/2025-01-5030

---

## Further practice

External repository (the author's companion workshop):
- [Notebook 01 — Sensor Overview](https://github.com/milinpatel07/AI-Perception-Safety-Workshop/blob/main/notebooks/01_sensor_overview.ipynb)
- [Notebook 12 — NHTSA Data Analysis](https://github.com/milinpatel07/AI-Perception-Safety-Workshop/blob/main/notebooks/12_nhtsa_data_analysis.ipynb)

---

**Navigation:** ← [Module 02 overview](README.md) · [Module index](README.md) · [Corner cases](02_corner_cases.ipynb) →